In [1]:
# Cell 1: Replicate 30D Excel VRP baseline in Python
#
# Objective:
#   Rebuild the current working 30D Excel-style VRP model in a clean Python branch.
#
# Scope:
#   - 30D only
#   - No Corsi model
#   - No cross-tenor optimization
#   - No Secondary
#   - No sizing
#
# Baseline specification:
#   Underlying:
#       SPY / SPX acceptable. Dividend issue ignored for this baseline.
#
#   Implied variance:
#       30D VIX-style implied variance.
#       Preferred source: internal ThetaData 30D VIX replication.
#
#   Realized variance:
#       RV21D = (STDEV.S(21D rolling close-to-close log returns) * sqrt(252))^2
#
#   VRP:
#       VRP = log(VIX^2 / RV21D)
#       Implemented as log(implied_variance_decimal / rv21d_variance_decimal)
#
#   Z-scores:
#       Prior-only.
#       Current VRP is NOT included in rolling mean/std.
#
#       3m:
#           window = 63 trading days
#           min_periods = 63
#           ddof = 1
#
#       1y:
#           window = 252 trading days
#           min_periods = 252
#           ddof = 1
#
#   Core signal:
#       RV21D vol > 8.5%
#       3m VRP z-score > 0.7
#       1y VRP z-score > 0.7
#       RSI14 < 70
#       VRP log > 0.7
#
# Outputs:
#   - Full 30D baseline signal tape
#   - Validation summary
#   - Performance summary
#   - Year-by-year performance
#   - Worst 20 selected trades
#   - Source manifest

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd


# =============================================================================
# 0. Constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BRANCH_NAME = "vrp_30d_baseline_v1"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BRANCH_NAME
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BRANCH_NAME
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / BRANCH_NAME

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

TENOR_TARGET = 30

TRADING_DAYS_PER_YEAR = 252
RV_WINDOW = 21
RV_DDOF = 1

Z3M_WINDOW = 63
Z3M_MIN_PERIODS = 63
Z1Y_WINDOW = 252
Z1Y_MIN_PERIODS = 252
Z_DDOF = 1

CORE_RV21D_VOL_MIN = 8.5
CORE_Z3M_MIN = 0.7
CORE_Z1Y_MIN = 0.7
CORE_RSI14_MAX = 70
CORE_VRP_LOG_MIN = 0.7

print("Cell 1: Replicate 30D Excel VRP baseline in Python")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Processed dir:      {PROCESSED_DIR}")
print(f"Audit dir:          {AUDIT_DIR}")
print(f"Notebook dir:       {NOTEBOOK_DIR}")
print(f"Run timestamp:      {RUN_TIMESTAMP}")

print()
print("Locked baseline conventions:")
display(pd.DataFrame([{
    "tenor_target": TENOR_TARGET,
    "rv_formula": "(STDEV.S(21D close-to-close log returns) * sqrt(252))^2",
    "rv_window": RV_WINDOW,
    "rv_ddof": RV_DDOF,
    "vrp_formula": "log(implied_variance_30d_decimal / rv21d_variance_decimal)",
    "z3m_window": Z3M_WINDOW,
    "z3m_min_periods": Z3M_MIN_PERIODS,
    "z1y_window": Z1Y_WINDOW,
    "z1y_min_periods": Z1Y_MIN_PERIODS,
    "z_ddof": Z_DDOF,
    "z_convention": "prior_only_shift1",
    "core_rv21d_vol_min": CORE_RV21D_VOL_MIN,
    "core_z3m_min": CORE_Z3M_MIN,
    "core_z1y_min": CORE_Z1Y_MIN,
    "core_rsi14_max": CORE_RSI14_MAX,
    "core_vrp_log_min": CORE_VRP_LOG_MIN,
}]))


# =============================================================================
# 1. Helpers
# =============================================================================

def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def first_existing_col(df, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def read_table(path, columns=None, nrows=None):
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        if columns is not None:
            return pd.read_parquet(path, columns=columns)
        return pd.read_parquet(path)

    if path.suffix.lower() in [".csv", ".txt"]:
        return pd.read_csv(path, usecols=columns, nrows=nrows)

    raise ValueError(f"Unsupported file extension: {path}")


def get_file_columns(path):
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        try:
            import pyarrow.parquet as pq
            return list(pq.read_schema(path).names)
        except Exception:
            return list(pd.read_parquet(path, nrows=0).columns)

    if path.suffix.lower() in [".csv", ".txt"]:
        return list(pd.read_csv(path, nrows=0).columns)

    return []


def safe_row_count(path):
    path = Path(path)

    try:
        if path.suffix.lower() == ".parquet":
            import pyarrow.parquet as pq
            return pq.ParquetFile(path).metadata.num_rows

        if path.suffix.lower() in [".csv", ".txt"]:
            # Slower, but only used for selected source files.
            return sum(1 for _ in open(path, "rb")) - 1
    except Exception:
        return np.nan

    return np.nan


def date_to_trade_date(series):
    return pd.to_datetime(series).dt.strftime("%Y%m%d").astype(int)


def compute_prior_only_z(series, window, min_periods, ddof):
    history = series.shift(1)
    rolling_mean = history.rolling(window=window, min_periods=min_periods).mean()
    rolling_std = history.rolling(window=window, min_periods=min_periods).std(ddof=ddof)
    z = (series - rolling_mean) / rolling_std
    return z.replace([np.inf, -np.inf], np.nan)


def profit_factor(values):
    values = pd.Series(values).dropna()

    if values.empty:
        return np.nan

    gains = values.loc[values > 0].sum()
    losses = values.loc[values < 0].sum()

    if losses < 0:
        return gains / abs(losses)

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def summarize_performance(df, signal_col, prefix):
    selected = df.loc[df[signal_col].astype(bool)].copy()

    if selected.empty:
        return {
            f"{prefix}_trades": 0,
            f"{prefix}_frequency": 0.0,
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_day_weighted_pnl_per_day": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade": np.nan,
            f"{prefix}_best_trade": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
        }

    total_dte = selected["actual_dte"].sum()

    return {
        f"{prefix}_trades": int(len(selected)),
        f"{prefix}_frequency": float(len(selected) / len(df)) if len(df) else np.nan,
        f"{prefix}_win_rate": float((selected["outcome_value"] > 0).mean()),
        f"{prefix}_avg_pnl_per_trade": float(selected["outcome_value"].mean()),
        f"{prefix}_avg_pnl_per_day": float(selected["pnl_per_day"].mean()),
        f"{prefix}_exposure_day_weighted_pnl_per_day": (
            float(selected["outcome_value"].sum() / total_dte) if total_dte > 0 else np.nan
        ),
        f"{prefix}_total_pnl": float(selected["outcome_value"].sum()),
        f"{prefix}_profit_factor": float(profit_factor(selected["outcome_value"])),
        f"{prefix}_worst_trade": float(selected["outcome_value"].min()),
        f"{prefix}_best_trade": float(selected["outcome_value"].max()),
        f"{prefix}_avg_actual_dte": float(selected["actual_dte"].mean()),
    }


def add_source_manifest_row(rows, role, path, df=None, columns_used=None, notes=None):
    path = Path(path)

    row = {
        "role": role,
        "path": str(path),
        "exists": path.exists(),
        "row_count_file": safe_row_count(path) if path.exists() else np.nan,
        "columns_used": json.dumps(columns_used or []),
        "notes": notes or "",
    }

    if df is not None and len(df):
        row.update({
            "row_count_loaded": len(df),
            "first_date_loaded": df["date"].min() if "date" in df.columns else pd.NaT,
            "last_date_loaded": df["date"].max() if "date" in df.columns else pd.NaT,
        })
    else:
        row.update({
            "row_count_loaded": np.nan,
            "first_date_loaded": pd.NaT,
            "last_date_loaded": pd.NaT,
        })

    rows.append(row)


# =============================================================================
# 2. Discover and load 30D implied vol / variance source
# =============================================================================

# Preferred sources are internal VIX-style tenor panels / signal panels that already contain
# vix_style_vol or implied_variance by date x tenor.
implied_search_dirs = [
    PROJECT_ROOT / "data" / "processed" / "forecast_model_corsi_v1",
    PROJECT_ROOT / "data" / "processed" / "corsi_signal_panel_v2",
    PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_prior_only_z",
    PROJECT_ROOT / "data" / "processed" / "corsi_parameter_sweep_clean",
    PROJECT_ROOT / "data" / "processed",
]

implied_file_candidates = []

for d in implied_search_dirs:
    if not d.exists():
        continue

    for path in sorted(list(d.glob("*.parquet")) + list(d.glob("*.csv"))):
        try:
            cols = get_file_columns(path)
        except Exception:
            continue

        has_tenor = any(c in cols for c in ["tenor", "target_tenor", "entry_tenor"])
        has_date = any(c in cols for c in ["date", "trade_date", "quote_date"])
        has_implied = any(c in cols for c in ["vix_style_vol", "vix_30d_vol", "implied_variance", "implied_var", "iv"])

        if has_tenor and has_date and has_implied:
            score = 0
            name = path.name.lower()
            parent = str(path.parent).lower()

            if "corsi_cell12_oos_forecast_vrp_signal_panel" in name:
                score += 100
            if "signal_panel" in name:
                score += 40
            if "parameter_sweep" in parent:
                score += 10
            if "vix" in name:
                score += 10
            if "vix_style_vol" in cols:
                score += 20
            if "implied_variance" in cols:
                score += 10

            implied_file_candidates.append({
                "path": path,
                "score": score,
                "columns": cols,
            })

if not implied_file_candidates:
    raise FileNotFoundError(
        "Could not find any implied-vol/variance panel with date, tenor, and vix_style_vol/implied_variance."
    )

implied_file_candidates = sorted(
    implied_file_candidates,
    key=lambda x: (x["score"], str(x["path"])),
    reverse=True,
)

IMPLIED_SOURCE_PATH = implied_file_candidates[0]["path"]

implied_raw = normalize_columns(read_table(IMPLIED_SOURCE_PATH))

date_col = first_existing_col(implied_raw, ["date", "quote_date"])
trade_date_col = first_existing_col(implied_raw, ["trade_date"])
tenor_col = first_existing_col(implied_raw, ["tenor", "target_tenor", "entry_tenor"])
vix_vol_col = first_existing_col(implied_raw, ["vix_style_vol", "vix_30d_vol"])
implied_var_col = first_existing_col(implied_raw, ["implied_variance", "implied_var"])

if tenor_col is None:
    raise ValueError(f"Implied source missing tenor column: {IMPLIED_SOURCE_PATH}")

if date_col is None and trade_date_col is None:
    raise ValueError(f"Implied source missing date/trade_date column: {IMPLIED_SOURCE_PATH}")

if vix_vol_col is None and implied_var_col is None:
    raise ValueError(f"Implied source missing vix_style_vol/implied_variance column: {IMPLIED_SOURCE_PATH}")

implied = implied_raw.copy()

if date_col is not None:
    implied["date"] = pd.to_datetime(implied[date_col]).dt.normalize()
elif trade_date_col is not None:
    implied["date"] = pd.to_datetime(implied[trade_date_col].astype(str), format="%Y%m%d").dt.normalize()

if trade_date_col is not None:
    implied["trade_date"] = pd.to_numeric(implied[trade_date_col], errors="coerce").astype("Int64")
else:
    implied["trade_date"] = date_to_trade_date(implied["date"])

implied["tenor"] = pd.to_numeric(implied[tenor_col], errors="coerce").astype("Int64")

implied = implied.loc[implied["tenor"].eq(TENOR_TARGET)].copy()

if implied.empty:
    raise ValueError(f"No rows found for tenor={TENOR_TARGET} in implied source: {IMPLIED_SOURCE_PATH}")

if vix_vol_col is not None:
    implied["vix_30d_vol_pct"] = pd.to_numeric(implied[vix_vol_col], errors="coerce")
    implied["implied_variance_decimal"] = (implied["vix_30d_vol_pct"] / 100.0) ** 2
    implied_variance_source_notes = f"Used {vix_vol_col} as percent vol."
else:
    implied_var = pd.to_numeric(implied[implied_var_col], errors="coerce")
    median_iv = implied_var.median()

    # Existing project implied_variance has usually been decimal variance,
    # e.g. 0.04 corresponds to 20 vol.
    if median_iv < 5:
        implied["implied_variance_decimal"] = implied_var
        implied["vix_30d_vol_pct"] = np.sqrt(implied["implied_variance_decimal"]) * 100.0
        implied_variance_source_notes = f"Used {implied_var_col} as decimal variance."
    else:
        implied["implied_variance_decimal"] = implied_var / (100.0 ** 2)
        implied["vix_30d_vol_pct"] = np.sqrt(implied_var)
        implied_variance_source_notes = f"Used {implied_var_col} as percent-vol-squared variance."

implied = (
    implied[["date", "trade_date", "tenor", "vix_30d_vol_pct", "implied_variance_decimal"]]
    .dropna(subset=["date", "trade_date", "tenor", "vix_30d_vol_pct", "implied_variance_decimal"])
    .drop_duplicates(["trade_date", "tenor"])
    .sort_values("date")
    .reset_index(drop=True)
)

print()
print("30D implied vol / variance source selected:")
print(IMPLIED_SOURCE_PATH)
print(implied_variance_source_notes)

print()
print("30D implied source summary:")
display(pd.DataFrame([{
    "rows": len(implied),
    "first_date": implied["date"].min(),
    "last_date": implied["date"].max(),
    "median_vix_30d_vol_pct": implied["vix_30d_vol_pct"].median(),
    "min_vix_30d_vol_pct": implied["vix_30d_vol_pct"].min(),
    "max_vix_30d_vol_pct": implied["vix_30d_vol_pct"].max(),
}]))


# =============================================================================
# 3. Discover and load close / RSI source
# =============================================================================

feature_search_dirs = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "interim",
]

feature_candidates = []

for d in feature_search_dirs:
    if not d.exists():
        continue

    for path in sorted(list(d.rglob("*.parquet")) + list(d.rglob("*.csv"))):
        # Skip obvious large options files unless they have useful fields.
        try:
            cols = get_file_columns(path)
        except Exception:
            continue

        has_date = any(c in cols for c in ["date", "trade_date", "quote_date"])
        has_close = any(c.lower() in [
            "close", "spy_close", "spx_close", "underlying_close",
            "adj_close", "adjusted_close", "close_price", "spy_adj_close"
        ] for c in cols)
        has_rsi = any(c.lower() in ["rsi14", "rsi_14", "rsi"] for c in cols)

        if has_date and (has_close or has_rsi):
            score = 0
            name = path.name.lower()

            if "production_feature_panel" in name:
                score += 100
            if has_close:
                score += 30
            if has_rsi:
                score += 20

            feature_candidates.append({
                "path": path,
                "score": score,
                "columns": cols,
                "has_close": has_close,
                "has_rsi": has_rsi,
            })

if not feature_candidates:
    raise FileNotFoundError("Could not find feature/price source with date and close or RSI columns.")

feature_candidates = sorted(
    feature_candidates,
    key=lambda x: (x["score"], str(x["path"])),
    reverse=True,
)

# Prefer one source with both close and RSI if available.
both_candidates = [c for c in feature_candidates if c["has_close"] and c["has_rsi"]]
FEATURE_SOURCE_PATH = both_candidates[0]["path"] if both_candidates else feature_candidates[0]["path"]

feature_raw = normalize_columns(read_table(FEATURE_SOURCE_PATH))

feature_date_col = first_existing_col(feature_raw, ["date", "quote_date"])
feature_trade_date_col = first_existing_col(feature_raw, ["trade_date"])

close_col = first_existing_col(
    feature_raw,
    [
        "spy_close",
        "spx_close",
        "underlying_close",
        "adj_close",
        "adjusted_close",
        "close",
        "close_price",
        "spy_adj_close",
    ],
)

rsi_col = first_existing_col(feature_raw, ["rsi14", "rsi_14", "rsi"])

if feature_date_col is None and feature_trade_date_col is None:
    raise ValueError(f"Feature source missing date/trade_date column: {FEATURE_SOURCE_PATH}")

feature = feature_raw.copy()

if feature_date_col is not None:
    feature["date"] = pd.to_datetime(feature[feature_date_col]).dt.normalize()
elif feature_trade_date_col is not None:
    feature["date"] = pd.to_datetime(feature[feature_trade_date_col].astype(str), format="%Y%m%d").dt.normalize()

if feature_trade_date_col is not None:
    feature["trade_date"] = pd.to_numeric(feature[feature_trade_date_col], errors="coerce").astype("Int64")
else:
    feature["trade_date"] = date_to_trade_date(feature["date"])

feature_out_cols = ["date", "trade_date"]

if close_col is not None:
    feature["underlying_close"] = pd.to_numeric(feature[close_col], errors="coerce")
    feature_out_cols.append("underlying_close")

if rsi_col is not None:
    feature["rsi14"] = pd.to_numeric(feature[rsi_col], errors="coerce")
    feature_out_cols.append("rsi14")

feature = (
    feature[feature_out_cols]
    .dropna(subset=["date", "trade_date"])
    .drop_duplicates("trade_date")
    .sort_values("date")
    .reset_index(drop=True)
)

if "underlying_close" not in feature.columns:
    raise ValueError(
        f"Feature source does not contain a close column. "
        f"Cannot compute Excel baseline RV21D from returns. Source: {FEATURE_SOURCE_PATH}"
    )

# Compute close-to-close log returns and Excel-style RV21D.
feature["daily_log_return"] = np.log(feature["underlying_close"] / feature["underlying_close"].shift(1))

feature["rv21d_vol_decimal"] = (
    feature["daily_log_return"]
    .rolling(window=RV_WINDOW, min_periods=RV_WINDOW)
    .std(ddof=RV_DDOF)
    * np.sqrt(TRADING_DAYS_PER_YEAR)
)

feature["rv21d_variance_decimal"] = feature["rv21d_vol_decimal"] ** 2
feature["rv21d_vol_pct"] = feature["rv21d_vol_decimal"] * 100.0

# If RSI is missing, compute a fallback. Prefer source RSI if present.
if "rsi14" not in feature.columns:
    delta = feature["underlying_close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder-style RSI.
    avg_gain = gain.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()

    rs = avg_gain / avg_loss
    feature["rsi14"] = 100 - (100 / (1 + rs))
    rsi_source_note = "Computed Wilder RSI14 fallback from close because source RSI was not found."
else:
    rsi_source_note = f"Used source RSI column: {rsi_col}"

feature = feature[
    [
        "date",
        "trade_date",
        "underlying_close",
        "daily_log_return",
        "rv21d_vol_decimal",
        "rv21d_variance_decimal",
        "rv21d_vol_pct",
        "rsi14",
    ]
].copy()

print()
print("Close / RSI source selected:")
print(FEATURE_SOURCE_PATH)
print(f"Close column used: {close_col}")
print(rsi_source_note)

print()
print("Close / RV / RSI source summary:")
display(pd.DataFrame([{
    "rows": len(feature),
    "first_date": feature["date"].min(),
    "last_date": feature["date"].max(),
    "close_non_null": int(feature["underlying_close"].notna().sum()),
    "log_return_non_null": int(feature["daily_log_return"].notna().sum()),
    "rv21d_non_null": int(feature["rv21d_variance_decimal"].notna().sum()),
    "rsi14_non_null": int(feature["rsi14"].notna().sum()),
    "median_rv21d_vol_pct": feature["rv21d_vol_pct"].median(),
    "min_rv21d_vol_pct": feature["rv21d_vol_pct"].min(),
    "max_rv21d_vol_pct": feature["rv21d_vol_pct"].max(),
}]))


# =============================================================================
# 4. Build 30D baseline signal panel
# =============================================================================

signal = implied.merge(
    feature,
    on=["date", "trade_date"],
    how="left",
    validate="one_to_one",
)

signal["vrp_log"] = np.log(signal["implied_variance_decimal"] / signal["rv21d_variance_decimal"])

signal = signal.sort_values("date").reset_index(drop=True)

signal["vrp_z_3m"] = compute_prior_only_z(
    signal["vrp_log"],
    window=Z3M_WINDOW,
    min_periods=Z3M_MIN_PERIODS,
    ddof=Z_DDOF,
)

signal["vrp_z_1y"] = compute_prior_only_z(
    signal["vrp_log"],
    window=Z1Y_WINDOW,
    min_periods=Z1Y_MIN_PERIODS,
    ddof=Z_DDOF,
)

signal["core_signal"] = (
    signal["rv21d_vol_pct"].gt(CORE_RV21D_VOL_MIN)
    & signal["vrp_z_3m"].gt(CORE_Z3M_MIN)
    & signal["vrp_z_1y"].gt(CORE_Z1Y_MIN)
    & signal["rsi14"].lt(CORE_RSI14_MAX)
    & signal["vrp_log"].gt(CORE_VRP_LOG_MIN)
)

# Formula audit fields.
signal["vrp_log_recalc"] = np.log(signal["implied_variance_decimal"] / signal["rv21d_variance_decimal"])
signal["vrp_log_recalc_diff"] = signal["vrp_log"] - signal["vrp_log_recalc"]

signal["vrp_z_3m_recalc"] = compute_prior_only_z(
    signal["vrp_log"],
    window=Z3M_WINDOW,
    min_periods=Z3M_MIN_PERIODS,
    ddof=Z_DDOF,
)

signal["vrp_z_1y_recalc"] = compute_prior_only_z(
    signal["vrp_log"],
    window=Z1Y_WINDOW,
    min_periods=Z1Y_MIN_PERIODS,
    ddof=Z_DDOF,
)

signal["vrp_z_3m_recalc_diff"] = signal["vrp_z_3m"] - signal["vrp_z_3m_recalc"]
signal["vrp_z_1y_recalc_diff"] = signal["vrp_z_1y"] - signal["vrp_z_1y_recalc"]


# =============================================================================
# 5. Discover and join 30D trade outcomes
# =============================================================================

outcome_search_dirs = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "audit",
]

outcome_candidates = []

for d in outcome_search_dirs:
    if not d.exists():
        continue

    for path in sorted(list(d.rglob("*.parquet")) + list(d.rglob("*.csv"))):
        try:
            cols = get_file_columns(path)
        except Exception:
            continue

        has_date = any(c in cols for c in ["trade_date", "date", "entry_date"])
        has_tenor = any(c in cols for c in ["tenor", "entry_tenor", "target_tenor"])
        has_pnl = any(c in cols for c in ["normalized_pnl_dollars", "outcome_value", "pnl", "pnl_dollars"])
        has_dte = any(c in cols for c in ["actual_dte"])

        if has_date and has_tenor and has_pnl and has_dte:
            score = 0
            name = path.name.lower()

            if "put_candidate_universe_pricing" in name:
                score += 100
            if "outcome" in name:
                score += 40
            if "normalized_pnl_dollars" in cols:
                score += 20
            if "actual_dte" in cols:
                score += 20

            outcome_candidates.append({
                "path": path,
                "score": score,
                "columns": cols,
            })

if not outcome_candidates:
    raise FileNotFoundError(
        "Could not find outcome file with trade_date/date, tenor, normalized_pnl_dollars/outcome, and actual_dte."
    )

outcome_candidates = sorted(
    outcome_candidates,
    key=lambda x: (x["score"], str(x["path"])),
    reverse=True,
)

OUTCOME_SOURCE_PATH = outcome_candidates[0]["path"]

outcome_raw = normalize_columns(read_table(OUTCOME_SOURCE_PATH))

outcome_date_col = first_existing_col(outcome_raw, ["trade_date", "date", "entry_date"])
outcome_tenor_col = first_existing_col(outcome_raw, ["tenor", "entry_tenor", "target_tenor"])
outcome_pnl_col = first_existing_col(outcome_raw, ["normalized_pnl_dollars", "outcome_value", "pnl", "pnl_dollars"])
actual_dte_col = first_existing_col(outcome_raw, ["actual_dte"])

if outcome_date_col is None or outcome_tenor_col is None or outcome_pnl_col is None or actual_dte_col is None:
    raise ValueError(f"Outcome source missing required columns: {OUTCOME_SOURCE_PATH}")

outcome = outcome_raw.copy()

if outcome_date_col == "trade_date":
    outcome["trade_date"] = pd.to_numeric(outcome[outcome_date_col], errors="coerce").astype("Int64")
    outcome["date"] = pd.to_datetime(outcome["trade_date"].astype(str), format="%Y%m%d", errors="coerce").dt.normalize()
else:
    outcome["date"] = pd.to_datetime(outcome[outcome_date_col], errors="coerce").dt.normalize()
    outcome["trade_date"] = date_to_trade_date(outcome["date"])

outcome["tenor"] = pd.to_numeric(outcome[outcome_tenor_col], errors="coerce").astype("Int64")
outcome["outcome_value"] = pd.to_numeric(outcome[outcome_pnl_col], errors="coerce")
outcome["actual_dte"] = pd.to_numeric(outcome[actual_dte_col], errors="coerce")

outcome = outcome.loc[outcome["tenor"].eq(TENOR_TARGET)].copy()

outcome = (
    outcome[["date", "trade_date", "tenor", "outcome_value", "actual_dte"]]
    .dropna(subset=["date", "trade_date", "tenor", "outcome_value", "actual_dte"])
    .drop_duplicates(["trade_date", "tenor"])
    .sort_values("date")
    .reset_index(drop=True)
)

signal = signal.merge(
    outcome[["trade_date", "tenor", "outcome_value", "actual_dte"]],
    on=["trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

signal["has_outcome"] = signal["outcome_value"].notna() & signal["actual_dte"].notna() & signal["actual_dte"].gt(0)
signal["pnl_per_day"] = signal["outcome_value"] / signal["actual_dte"]

print()
print("Outcome source selected:")
print(OUTCOME_SOURCE_PATH)
print(f"Outcome P&L column used: {outcome_pnl_col}")
print(f"Actual DTE column used:  {actual_dte_col}")

print()
print("Outcome source summary:")
display(pd.DataFrame([{
    "rows_30d": len(outcome),
    "first_date": outcome["date"].min(),
    "last_date": outcome["date"].max(),
    "outcome_non_null": int(outcome["outcome_value"].notna().sum()),
    "actual_dte_non_null": int(outcome["actual_dte"].notna().sum()),
    "median_actual_dte": outcome["actual_dte"].median(),
}]))


# =============================================================================
# 6. Validation checks
# =============================================================================

validation_rows = []

def add_validation(check_name, passed, details):
    validation_rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details,
    })

add_validation(
    "30D implied vol source found",
    len(implied) > 0 and signal["vix_30d_vol_pct"].notna().any(),
    f"path={IMPLIED_SOURCE_PATH}; rows_30d={len(implied)}",
)

add_validation(
    "Close source found",
    signal["underlying_close"].notna().any(),
    f"path={FEATURE_SOURCE_PATH}; close_col={close_col}",
)

add_validation(
    "RV21D formula available",
    signal["rv21d_variance_decimal"].notna().any(),
    f"rv_window={RV_WINDOW}; ddof={RV_DDOF}; annualization={TRADING_DAYS_PER_YEAR}",
)

add_validation(
    "VRP formula audit",
    signal["vrp_log_recalc_diff"].dropna().abs().max() == 0,
    f"max_abs_diff={signal['vrp_log_recalc_diff'].dropna().abs().max()}",
)

add_validation(
    "3m z-score prior-only audit",
    signal["vrp_z_3m_recalc_diff"].dropna().abs().max() == 0,
    f"window={Z3M_WINDOW}; min_periods={Z3M_MIN_PERIODS}; ddof={Z_DDOF}; max_abs_diff={signal['vrp_z_3m_recalc_diff'].dropna().abs().max()}",
)

add_validation(
    "1y z-score prior-only audit",
    signal["vrp_z_1y_recalc_diff"].dropna().abs().max() == 0,
    f"window={Z1Y_WINDOW}; min_periods={Z1Y_MIN_PERIODS}; ddof={Z_DDOF}; max_abs_diff={signal['vrp_z_1y_recalc_diff'].dropna().abs().max()}",
)

add_validation(
    "RSI14 available",
    signal["rsi14"].notna().any(),
    rsi_source_note,
)

add_validation(
    "Outcome join",
    signal["outcome_value"].notna().any(),
    f"path={OUTCOME_SOURCE_PATH}; outcome_col={outcome_pnl_col}; joined_non_null={int(signal['outcome_value'].notna().sum())}",
)

add_validation(
    "Actual DTE join",
    signal["actual_dte"].notna().any() and signal.loc[signal["actual_dte"].notna(), "actual_dte"].gt(0).all(),
    f"actual_dte_col={actual_dte_col}; joined_non_null={int(signal['actual_dte'].notna().sum())}",
)

add_validation(
    "P&L/day formula",
    (
        (signal.loc[signal["has_outcome"], "pnl_per_day"]
         - signal.loc[signal["has_outcome"], "outcome_value"] / signal.loc[signal["has_outcome"], "actual_dte"])
        .abs()
        .max()
        == 0
    ) if signal["has_outcome"].any() else False,
    "pnl_per_day = outcome_value / actual_dte",
)

validation = pd.DataFrame(validation_rows)

print()
print("Validation checks:")
display(validation)

if validation["status"].eq("FAIL").any():
    print()
    print("FAILED VALIDATION CHECKS:")
    display(validation.loc[validation["status"].eq("FAIL")])
    raise ValueError("Validation failed. Stop before reviewing performance.")


# =============================================================================
# 7. Baseline signal / performance
# =============================================================================

required_signal_cols = [
    "vix_30d_vol_pct",
    "implied_variance_decimal",
    "rv21d_vol_pct",
    "rv21d_variance_decimal",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
]

signal["is_baseline_feature_complete"] = signal[required_signal_cols].notna().all(axis=1)
signal["is_baseline_trade_eligible"] = signal["is_baseline_feature_complete"] & signal["has_outcome"]

baseline_trade_panel = signal.loc[signal["is_baseline_trade_eligible"]].copy()

performance_summary = pd.DataFrame([
    {
        "panel": "all_30d_rows",
        "rows": len(signal),
        "first_date": signal["date"].min(),
        "last_date": signal["date"].max(),
        "feature_complete_rows": int(signal["is_baseline_feature_complete"].sum()),
        "trade_eligible_rows": int(signal["is_baseline_trade_eligible"].sum()),
        **summarize_performance(signal.loc[signal["is_baseline_trade_eligible"]], "core_signal", "core"),
    }
])

year_summary = (
    baseline_trade_panel
    .assign(year=baseline_trade_panel["date"].dt.year)
    .groupby("year")
    .apply(lambda g: pd.Series({
        "eligible_rows": len(g),
        "core_trades": int(g["core_signal"].sum()),
        "core_frequency": float(g["core_signal"].mean()),
        "core_win_rate": float((g.loc[g["core_signal"], "outcome_value"] > 0).mean()) if g["core_signal"].any() else np.nan,
        "core_avg_pnl_per_trade": float(g.loc[g["core_signal"], "outcome_value"].mean()) if g["core_signal"].any() else np.nan,
        "core_avg_pnl_per_day": float(g.loc[g["core_signal"], "pnl_per_day"].mean()) if g["core_signal"].any() else np.nan,
        "core_total_pnl": float(g.loc[g["core_signal"], "outcome_value"].sum()) if g["core_signal"].any() else 0.0,
        "core_worst_trade": float(g.loc[g["core_signal"], "outcome_value"].min()) if g["core_signal"].any() else np.nan,
    }))
    .reset_index()
)

worst_20_trades = (
    baseline_trade_panel
    .loc[baseline_trade_panel["core_signal"]]
    .sort_values("outcome_value", ascending=True)
    .head(20)
    [
        [
            "date",
            "trade_date",
            "tenor",
            "vix_30d_vol_pct",
            "rv21d_vol_pct",
            "vrp_log",
            "vrp_z_3m",
            "vrp_z_1y",
            "rsi14",
            "outcome_value",
            "actual_dte",
            "pnl_per_day",
        ]
    ]
    .copy()
)

best_20_trades = (
    baseline_trade_panel
    .loc[baseline_trade_panel["core_signal"]]
    .sort_values("outcome_value", ascending=False)
    .head(20)
    [
        [
            "date",
            "trade_date",
            "tenor",
            "vix_30d_vol_pct",
            "rv21d_vol_pct",
            "vrp_log",
            "vrp_z_3m",
            "vrp_z_1y",
            "rsi14",
            "outcome_value",
            "actual_dte",
            "pnl_per_day",
        ]
    ]
    .copy()
)

print()
print("Baseline 30D signal availability:")
display(pd.DataFrame([{
    "rows": len(signal),
    "first_date": signal["date"].min(),
    "last_date": signal["date"].max(),
    "feature_complete_rows": int(signal["is_baseline_feature_complete"].sum()),
    "trade_eligible_rows": int(signal["is_baseline_trade_eligible"].sum()),
    "core_signal_rows_all_dates": int(signal["core_signal"].sum()),
    "core_signal_rows_trade_eligible": int(baseline_trade_panel["core_signal"].sum()),
}]))


print()
print("Baseline 30D Core performance:")
display(performance_summary)

print()
print("Baseline 30D Core performance by year:")
display(year_summary)

print()
print("Worst 20 selected Core trades:")
display(worst_20_trades)

print()
print("Best 20 selected Core trades:")
display(best_20_trades)


# =============================================================================
# 8. Source manifest
# =============================================================================

manifest_rows = []

add_source_manifest_row(
    manifest_rows,
    role="30d_implied_vol_variance",
    path=IMPLIED_SOURCE_PATH,
    df=implied,
    columns_used=["date", "trade_date", "tenor", vix_vol_col or implied_var_col],
    notes=implied_variance_source_notes,
)

add_source_manifest_row(
    manifest_rows,
    role="close_rsi_features",
    path=FEATURE_SOURCE_PATH,
    df=feature,
    columns_used=["date", "trade_date", close_col, rsi_col],
    notes=rsi_source_note,
)

add_source_manifest_row(
    manifest_rows,
    role="trade_outcomes",
    path=OUTCOME_SOURCE_PATH,
    df=outcome,
    columns_used=["date", "trade_date", "tenor", outcome_pnl_col, actual_dte_col],
    notes="30D outcomes only",
)

source_manifest = pd.DataFrame(manifest_rows)

print()
print("Source manifest:")
display(source_manifest)


# =============================================================================
# 9. Save outputs
# =============================================================================

safe_start = signal["date"].min().strftime("%Y%m%d")
safe_end = signal["date"].max().strftime("%Y%m%d")

signal_tape_path = (
    PROCESSED_DIR
    / f"00_excel_30d_vrp_baseline_signal_tape_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

signal_tape_csv_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_signal_tape_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

validation_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

performance_summary_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_performance_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

year_summary_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_year_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

worst_20_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_worst_20_trades_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

best_20_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_best_20_trades_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

manifest_path = (
    AUDIT_DIR
    / f"00_excel_30d_vrp_baseline_source_manifest_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

signal.to_parquet(signal_tape_path, index=False)
signal.to_csv(signal_tape_csv_path, index=False)

validation.to_csv(validation_path, index=False)
performance_summary.to_csv(performance_summary_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
worst_20_trades.to_csv(worst_20_path, index=False)
best_20_trades.to_csv(best_20_path, index=False)
source_manifest.to_csv(manifest_path, index=False)

print()
print("Cell 1 outputs saved:")
print(f"Signal tape parquet:      {signal_tape_path}")
print(f"Signal tape CSV:          {signal_tape_csv_path}")
print(f"Validation checks:        {validation_path}")
print(f"Performance summary:      {performance_summary_path}")
print(f"Year summary:             {year_summary_path}")
print(f"Worst 20 trades:          {worst_20_path}")
print(f"Best 20 trades:           {best_20_path}")
print(f"Source manifest:          {manifest_path}")

print()
print("Cell 1 complete.")

Cell 1: Replicate 30D Excel VRP baseline in Python
Project root:       C:\Users\patri\vrp_project
Processed dir:      C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1
Audit dir:          C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1
Notebook dir:       C:\Users\patri\vrp_project\notebooks\vrp_30d_baseline_v1
Run timestamp:      20260704_122658

Locked baseline conventions:


,tenor_target,rv_formula,rv_window,rv_ddof,vrp_formula,z3m_window,z3m_min_periods,z1y_window,z1y_min_periods,z_ddof,z_convention,core_rv21d_vol_min,core_z3m_min,core_z1y_min,core_rsi14_max,core_vrp_log_min
0,30,(STDEV.S(21D close-to-close log returns) * sqr...,21,1,log(implied_variance_30d_decimal / rv21d_varia...,63,63,252,252,1,prior_only_shift1,8.5,0.7,0.7,70,0.7



30D implied vol / variance source selected:
C:\Users\patri\vrp_project\data\processed\forecast_model_corsi_v1\corsi_cell12_oos_forecast_vrp_signal_panel_20200102_20260623_utp_cta_20260703_211350.parquet
Used vix_style_vol as percent vol.

30D implied source summary:


,rows,first_date,last_date,median_vix_30d_vol_pct,min_vix_30d_vol_pct,max_vix_30d_vol_pct
0,1612,2020-01-02,2026-06-02,18.972151,11.871154,80.414657



Close / RSI source selected:
C:\Users\patri\vrp_project\data\processed\staging\production_feature_panel_v0_1_candidate_20180625_20260702.parquet
Close column used: spx_close
Used source RSI column: rsi14

Close / RV / RSI source summary:


...


Best 20 selected Core trades:


,date,trade_date,tenor,vix_30d_vol_pct,rv21d_vol_pct,vrp_log,vrp_z_3m,vrp_z_1y,rsi14,outcome_value,actual_dte,pnl_per_day
1323,2025-04-08,20250408,30,52.759207,31.274179,1.045891,1.671586,0.913384,21.362549,45095.398744,31.0,1454.690282
689,2022-09-27,20220927,30,32.689829,21.447283,0.842932,2.704420,0.778227,26.859738,32380.205577,31.0,1044.522761
522,2022-01-27,20220127,30,31.035304,13.231284,1.705083,1.129581,1.512602,24.320664,29954.859691,29.0,1032.926196
520,2022-01-25,20220125,30,30.575075,14.714719,1.462674,0.852340,1.061129,25.808804,29863.765222,31.0,963.347265
521,2022-01-26,20220126,30,31.092443,13.293846,1.699327,1.158403,1.504908,25.488450,28965.983361,30.0,965.532779
1154,2024-08-05,20240805,30,38.022912,19.039527,1.383344,1.458239,2.095828,30.554982,28845.059994,32.0,901.408125
483,2021-12-01,20211201,30,32.426059,13.553479,1.744638,0.911058,1.372853,37.494900,28384.414940,30.0,946.147165
269,2021-01-27,20210127,30,32.372845,14.802947,1.564987,0.793045,1.315038,47.275350,28367.508538,30.0,945.583618
1567,2026-03-30,20260330,30,30.339952,14.684497,1.451346,1.455426,1.025803,27.717242,28185.354965,32.0,880.792343
1566,2026-03-27,20260327,30,30.888215,14.685464,1.487033,1.646424,1.084017,28.658202,27555.995195,28.0,984.142686



Source manifest:


,role,path,exists,row_count_file,columns_used,notes,row_count_loaded,first_date_loaded,last_date_loaded
0,30d_implied_vol_variance,C:\Users\patri\vrp_project\data\processed\fore...,True,14565,"[""date"", ""trade_date"", ""tenor"", ""vix_style_vol""]",Used vix_style_vol as percent vol.,1612,2020-01-02,2026-06-02
1,close_rsi_features,C:\Users\patri\vrp_project\data\processed\stag...,True,18144,"[""date"", ""trade_date"", ""spx_close"", ""rsi14""]",Used source RSI column: rsi14,2016,2018-06-25,2026-07-02
2,trade_outcomes,C:\Users\patri\vrp_project\data\processed\put_...,True,18099,"[""date"", ""trade_date"", ""tenor"", ""normalized_pn...",30D outcomes only,1989,2018-06-25,2026-05-22



Cell 1 outputs saved:
Signal tape parquet:      C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_signal_tape_20200102_20260602_20260704_122658.parquet
Signal tape CSV:          C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_signal_tape_20200102_20260602_20260704_122658.csv
Validation checks:        C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_validation_20200102_20260602_20260704_122658.csv
Performance summary:      C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_performance_summary_20200102_20260602_20260704_122658.csv
Year summary:             C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_year_summary_20200102_20260602_20260704_122658.csv
Worst 20 trades:          C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_worst_20_trades_20200102_20260602_20260704_122658.c